# **Installation**

In [ ]:
pip install pandas

In [ ]:
!pip install matplotlib seaborn plotly
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [2]:
!pip install duckdb duckdb-engine jupysql --quiet

import duckdb
%reload_ext sql
%sql duckdb:///:memory:

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 12.8 MB/s eta 0:00:00


Connecting to 'duckdb:///:memory:'

In [3]:
#Once csv loaded in Collab - run this cell

# 1. Connexion à une base en mémoire
conn = duckdb.connect(database=':memory:')

# 2. Création de table SQL à partir des CSV
conn.execute("CREATE TABLE dropout AS SELECT * FROM read_csv_auto('clinical_trial_dataset_1000(1).csv')")

# 3. Configurer JupySQL
%reload_ext sql
%config SqlMagic.autopandas = True
%sql conn

In [ ]:
# 1. CRÉER BASE SQLITE directement dans Colab
import sqlite3
import pandas as pd

# Ta requête SQL → directement DataFrame
query = """
SELECT
  Gender, Age, Health_Condition, Trial_Phase, previous_adherence_score, Dropout_Flag
FROM clinical_dropout.clinical_trial_dataset_1000(1)
"""

# Si tu as accès BigQuery
print("Méthode 1 : SQLite en mémoire - Prêt !")

Méthode 1 : SQLite en mémoire - Prêt !


In [ ]:
!pip install pandas scikit-learn joblib

##upload CSV file

In [4]:
from google.colab import files
uploaded = files.upload()

Saving clinical_trial_dataset_1000(1).csv to clinical_trial_dataset_1000(1) (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("clinical_trial_dataset_1000(1).csv")
print("Dataset loaded")
df.head()

Dataset loaded


,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,NaN,Arthritis,0.85,Phase I,No,NaN
1,P0002,32,Male,White,NaN,NaN,0.72,Phase I,No,NaN
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,Yes,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,No,NaN
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,Yes,Medical Advice


# **SQL Coding**

In [5]:
%%sql

SELECT * FROM dropout

Running query in 'DuckDBPyConnection'

,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,None,Arthritis,0.85,Phase I,False,None
1,P0002,32,Male,White,None,None,0.72,Phase I,False,None
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,True,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,False,None
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,True,Medical Advice
...,...,...,...,...,...,...,...,...,...,...
995,P0996,44,Male,Asian,None,None,0.72,Phase I,True,Medical Advice
996,P0997,54,Female,Asian,Hypertension,Arthritis,0.76,Phase II,True,Logistical Issues
997,P0998,61,Male,Hispanic,None,Kidney Disease,0.98,Phase I,False,None
998,P0999,42,Male,Asian,Heart Disease,Depression,0.56,Phase I,True,None


In [ ]:
%%sql

-- TOTAL GLOBAL DROPOUT PARTICIPANTS
SELECT DISTINCT
  COUNT(*) AS Total_participants,
  SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout, -- dropout = 1
  CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') As dropout_rate
FROM dropout

Running query in 'DuckDBPyConnection'

,Total_participants,nb_dropout,dropout_rate
0,1000,262.0,26.2%


In [ ]:
%%sql

-- TOTAL DROPOUT BY GENDER
  SELECT DISTINCT
    Gender,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY Gender,
  ORDER BY rank

  -- on remarque 2x plus de femmes participantes (616 vs 384) => Possibilité de l'impact du déséquilibre de la taille d’échantillon 'Female Gender' à surveiller... si regression logistique utiliser une pondération

Running query in 'DuckDBPyConnection'

,Gender,nb_participants,nb_dropout,Dropout_rate,rank
0,Male,384,101.0,26.3%,1
1,Female,616,161.0,26.14%,2


In [14]:
%%sql

-- TOTAL DROPOUT BY AGE
WITH ranked_dropout AS (
  SELECT
    Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
)
SELECT
  rank,
  age_group,
  dropout_rate
FROM ranked_dropout
ORDER BY rank ASC
LIMIT 3

Running query in 'DuckDBPyConnection'

,rank,age_group,dropout_rate
0,1,80-89,42.11%
1,2,20-29,29.11%
2,3,10-19,27.78%


In [13]:
%%sql

-- TOTAL DROPOUT BY HEALTH_CONDITION & COMORBIDITIES
WITH ranked_dropout AS (
  SELECT
    Health_Condition,
    Comorbidities,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_Rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
  FROM dropout
  GROUP BY Health_Condition, Comorbidities
)
SELECT
  Rank,
  Health_Condition,
  Comorbidities,
  Dropout_Rate,
FROM ranked_dropout
ORDER BY Rank
LIMIT 3

Running query in 'DuckDBPyConnection'

,Rank,Health_Condition,Comorbidities,Dropout_Rate
0,1,None,None,40.63%
1,2,Asthma,Arthritis,40.0%
2,3,Hypertension,Arthritis,39.47%


In [17]:
%%sql

-- TOTAL DROPOUT BY GENDER / AGE / HEALTH_CONDITION & COMORBIDITIES
WITH ranked_dropout AS (
  SELECT
    gender,
    Comorbidities,
    Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS Age_Group,
    Health_Condition,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_Rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
  FROM dropout
  GROUP BY gender, Comorbidities, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END, Health_Condition
  HAVING COUNT(*) >= 5
)
SELECT
  Rank,
  gender,
  age_group,
  Comorbidities,
  Health_Condition,
  Dropout_Rate
FROM ranked_dropout
ORDER BY Rank ASC
LIMIT 5

Running query in 'DuckDBPyConnection'

,Rank,Gender,Age_Group,Comorbidities,Health_Condition,Dropout_Rate
0,1,Female,20-29,Arthritis,Asthma,71.43%
1,2,Female,60-69,None,Asthma,66.67%
2,3,Male,60-69,Arthritis,Heart Disease,60.0%
3,3,Female,40-49,None,None,60.0%
4,3,Female,40-49,Arthritis,Hypertension,60.0%


In [25]:
%%sql

-- TOTAL DROPOUT BY AGE / HEALTH_CONDITION & COMORBIDITIES
WITH Top_risk AS (
  SELECT DISTINCT
    Case FLOOR(Age/10)
    WHEN 2 THEN '20-29'
    WHEN 8 THEN '80-89'
    END AS Age_Group,
    Comorbidities,
    Health_Condition,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_Rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
  FROM dropout
  WHERE Age BETWEEN 20 AND 29 OR Age >= 80
  GROUP BY CASE FLOOR(Age/10) WHEN 2 THEN '20-29' WHEN 8 THEN '80-89' END, Comorbidities, Health_Condition
)
SELECT *
FROM Top_risk
ORDER BY Rank, nb_dropout DESC

Running query in 'DuckDBPyConnection'

,Age_Group,Comorbidities,Health_Condition,nb_participants,nb_dropout,Dropout_Rate,Rank
0,80-89,Arthritis,Diabetes,2,2.0,100.0%,1
1,80-89,Kidney Disease,Hypertension,1,1.0,100.0%,1
2,80-89,Obesity,Asthma,1,1.0,100.0%,1
3,20-29,Depression,Asthma,5,4.0,80.0%,2
4,20-29,Arthritis,Asthma,7,5.0,71.43%,3
5,20-29,Depression,None,6,4.0,66.67%,4
6,80-89,Depression,Hypertension,3,2.0,66.67%,4
7,20-29,None,Asthma,6,3.0,50.0%,5
8,20-29,Obesity,Asthma,4,2.0,50.0%,5
9,20-29,Depression,Cancer,4,2.0,50.0%,5


# Dropout Phases Analysis (PHASE 1, PHASE 2 or PHASE 3)

In [22]:
%%sql

-- ALL PARTICIPANTS RANK BY PHASES
  SELECT DISTINCT
    Trial_Phase,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_Rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
  FROM dropout
  GROUP BY Trial_Phase
  ORDER BY Rank, Dropout_Rate DESC

Running query in 'DuckDBPyConnection'

,Trial_Phase,nb_participants,nb_dropout,Dropout_Rate,Rank
0,Phase I,326,87.0,26.69%,1
1,Phase III,306,81.0,26.47%,2
2,Phase II,368,94.0,25.54%,3


In [23]:
%%sql

-- COUNTNG ALL PARTICIPANTS IN PHASE 1 BY GENDER
SELECT DISTINCT
Trial_Phase,
Gender,
COUNT(*) AS nb_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') AS Dropout_Rate,
DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
FROM dropout
WHERE Trial_Phase = 'Phase I'
GROUP BY Trial_Phase, Gender
ORDER BY Rank, Dropout_rate DESC

Running query in 'DuckDBPyConnection'

,Trial_Phase,Gender,nb_participants,nb_dropout,Dropout_Rate,Rank
0,Phase I,Female,208,57.0,27.4%,1
1,Phase I,Male,118,30.0,25.42%,2


In [30]:
%%sql

-- TOP 3 PARTICIPANTS IN PHASE 1 BY AGE
SELECT DISTINCT
Trial_Phase,
Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
COUNT(*) AS nb_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS Nb_Dropout,
CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') AS Dropout_Date,
DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS Rank
FROM dropout
WHERE Trial_Phase = 'Phase I'
GROUP BY Trial_Phase, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
ORDER BY Rank ASC
LIMIT 3

Running query in 'DuckDBPyConnection'

,Trial_Phase,age_group,nb_participants,Nb_Dropout,Dropout_Date,Rank
0,Phase I,10-19,5,3.0,60.0%,1
1,Phase I,80-89,5,2.0,40.0%,2
2,Phase I,20-29,51,18.0,35.29%,3


In [ ]:
%%sql

-- ALL PARTICIPANTS IN PHASE 1 BY HEALTH_CONDITION & COMORBIDITIES
SELECT DISTINCT
Trial_Phase,
Health_Condition,
Comorbidities,
COUNT(*) AS nb_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') AS dropout_rate,
DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
FROM dropout
WHERE Trial_Phase = 'Phase I'
GROUP BY Trial_Phase, Comorbidities, Health_Condition
ORDER BY rank, nb_participants DESC
LIMIT 5

Running query in 'DuckDBPyConnection'

,Trial_Phase,Health_Condition,Comorbidities,nb_participants,nb_dropout,dropout_rate,rank
0,Phase I,Heart Disease,Depression,7,4.0,57.14%,1
1,Phase I,Asthma,Arthritis,17,9.0,52.94%,2
2,Phase I,Cancer,Depression,8,4.0,50.0%,3
3,Phase I,Asthma,Depression,9,4.0,44.44%,4
4,Phase I,Hypertension,Arthritis,16,7.0,43.75%,5


In [ ]:
%%sql

-- ZOOM INTO PHASE 1/ 20-29 YEARS PARTICIPANTS BY GENDER
SELECT
  'Phase I 20-29 years' AS target_population,
  Gender,
  COUNT(*) AS total_participants,
  SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS total_dropouts,
  CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2), '%') AS dropout_rate
FROM dropout
WHERE Trial_Phase = 'Phase I'
  AND Age BETWEEN 20 AND 29
GROUP BY Gender
ORDER BY dropout_rate DESC

Running query in 'DuckDBPyConnection'

,target_population,Gender,total_participants,total_dropouts,dropout_rate
0,Phase I 20-29 years,Female,36,13.0,36.11%
1,Phase I 20-29 years,Male,15,5.0,33.33%


In [31]:
%%sql

-- ZOOM INTO PHASE 1/ 20-29 YEARS PARTICIPANTS BY HEALTH CONDITION & COMORBIDITIES
SELECT
  'Phase I 20-29 years' AS target_population,
  Health_Condition,
  Comorbidities,
  COUNT(*) AS total_participants,
  SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS total_dropouts,
  CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2), '%') AS dropout_rate,
  DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
FROM dropout
WHERE Trial_Phase = 'Phase I'
  AND Age BETWEEN 20 AND 29
GROUP BY Health_Condition, Comorbidities
ORDER BY rank, total_participants DESC
LIMIT 12

Running query in 'DuckDBPyConnection'

,target_population,Health_Condition,Comorbidities,total_participants,total_dropouts,dropout_rate,rank
0,Phase I 20-29 years,Asthma,Depression,3,3.0,100.0%,1
1,Phase I 20-29 years,Heart Disease,Arthritis,1,1.0,100.0%,1
2,Phase I 20-29 years,Heart Disease,Kidney Disease,1,1.0,100.0%,1
3,Phase I 20-29 years,None,Kidney Disease,1,1.0,100.0%,1
4,Phase I 20-29 years,Hypertension,Obesity,1,1.0,100.0%,1
5,Phase I 20-29 years,Asthma,Obesity,1,1.0,100.0%,1
6,Phase I 20-29 years,Asthma,Arthritis,4,3.0,75.0%,2
7,Phase I 20-29 years,Hypertension,Depression,6,3.0,50.0%,3
8,Phase I 20-29 years,None,Depression,2,1.0,50.0%,3
9,Phase I 20-29 years,Heart Disease,Depression,2,1.0,50.0%,3


In [ ]:
%%sql

-- ZOOM INTO PHASE 1/ 20-29 YEARS PARTICIPANTS BY DROPOUT REASON
SELECT
  'Phase I 20-29 years' AS target_population,
   Dropout_Reason,
  COUNT(*) AS total_participants,
  SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS total_dropouts,
  CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2), '%') AS dropout_rate,
  DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
FROM dropout
WHERE Trial_Phase = 'Phase I'
  AND Age BETWEEN 20 AND 29
GROUP BY Dropout_Reason
ORDER BY rank, total_participants DESC

Running query in 'DuckDBPyConnection'

,target_population,Dropout_Reason,total_participants,total_dropouts,dropout_rate,rank
0,Phase I 20-29 years,Lost Interest,4,4.0,100.0%,1
1,Phase I 20-29 years,Medical Advice,3,3.0,100.0%,1
2,Phase I 20-29 years,Side Effects,3,3.0,100.0%,1
3,Phase I 20-29 years,Logistical Issues,2,2.0,100.0%,1
4,Phase I 20-29 years,Lack of Improvement,1,1.0,100.0%,1
5,Phase I 20-29 years,None,38,5.0,13.16%,2


# **PREDICTIVE SCORING**

In [6]:
%%sql

-- VERIFICATION DE LA PLAGE DES DONNEES(MIN/MAX) DES ADHERENCES SCORES
SELECT DISTINCT
   ROUND(AVG(previous_adherence_score), 3) AS avg_adherence_global_score,
   MIN(previous_adherence_score) AS min_score,
   MAX(previous_adherence_score) AS max_score,
   COUNT(*) AS total_participants
FROM dropout

Running query in 'DuckDBPyConnection'

,avg_adherence_global_score,min_score,max_score,total_participants
0,0.75,0.1,1.0,1000


In [33]:
%%sql

--TOP 5 PREDICTIVE RISKS PROFILS BY SCORING (RANDOM FOREST)

WITH risk_with_adherence AS (
  SELECT
    Gender,
    CASE FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS Age_Group,
    Health_Condition,
    Comorbidities,
    AVG(previous_adherence_score) AS Avg_Score,
    COUNT(*) AS Total_Participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS Nb_Dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_Rate,
    DENSE_RANK() OVER (ORDER BY (1-AVG(previous_adherence_score))*30 +  -- Mauvaise adhérence passée = +risque
      (SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END)/COUNT(*))*70 DESC) AS Rank -- Dropout actuel
  FROM dropout
  GROUP BY Comorbidities, Gender, Health_Condition, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
)
SELECT
  Rank, Gender, Age_Group, Health_Condition, Comorbidities, Dropout_Rate,
  ROUND(Avg_Score, 3) AS Avg_Score,
  CASE
    WHEN Avg_Score BETWEEN 0 AND 0.4 THEN '🚨 HIGHT'
    WHEN Avg_Score BETWEEN 0.41 AND 0.7 THEN '⚠️  MEDIUM'
    ELSE '✅ LOW'
  END AS Adherence_Status
FROM risk_with_adherence
ORDER BY Rank
LIMIT 10

Running query in 'DuckDBPyConnection'

,Rank,Gender,Age_Group,Health_Condition,Comorbidities,Dropout_Rate,Avg_Score,Adherence_Status
0,1,Male,60-69,Asthma,Kidney Disease,100.0%,0.35,🚨 HIGHT
1,2,Male,20-29,Hypertension,Kidney Disease,100.0%,0.55,⚠️ MEDIUM
2,2,Male,50-59,None,Obesity,100.0%,0.55,⚠️ MEDIUM
3,3,Female,70-79,Heart Disease,Obesity,100.0%,0.56,⚠️ MEDIUM
4,4,Female,60-69,Heart Disease,None,100.0%,0.59,⚠️ MEDIUM
5,4,Male,60-69,None,None,100.0%,0.59,⚠️ MEDIUM
6,4,Male,80-89,None,None,100.0%,0.59,⚠️ MEDIUM
7,5,Male,80-89,Hypertension,Kidney Disease,100.0%,0.61,⚠️ MEDIUM
8,5,Male,60-69,Asthma,Arthritis,100.0%,0.61,⚠️ MEDIUM
9,6,Male,50-59,Heart Disease,Depression,100.0%,0.63,⚠️ MEDIUM
